# Prompt Evaluation

In [1]:
# Load env variables and create client
from anthropic import Anthropic
from dotenv import load_dotenv
import json

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

## Messages and Chat

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

## Generate Dataset

In [3]:
def generate_dataset():
    prompt = """Generate a evaluation dataset for a prompt evaluation. The dataset will be used to 
            evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related 
            tasks. Generate an array of JSON objects, each representing task that requires Python, 
            JSON, or a Regex to complete.

            Example output:
            ```json
            [
                {
                    "task": "Description of task",
                },
                ...additional
            ]
            ```

            * Focus on tasks that can be solved by writing a single Python function, a single JSON 
            object, or a regular expression.
            * Focus on tasks that do not require writing much code

            Please generate 3 objects."""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
dataset = generate_dataset()
print(json.dumps(dataset, indent=2))
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

[
  {
    "task": "Write a Python function that parses an AWS CloudWatch log entry and extracts the timestamp, log level, and message. The log format is: '[TIMESTAMP] [LEVEL] Message content here'"
  },
  {
    "task": "Create a JSON object that represents an AWS IAM policy allowing read-only access to S3 buckets with names starting with 'audit-' and DynamoDB tables named 'logs'"
  },
  {
    "task": "Write a regular expression that matches valid AWS ARN (Amazon Resource Name) formats. ARNs follow the pattern: arn:partition:service:region:account-id:resource-type/resource-id"
  }
]


## Run Prompt

In [5]:
def run_prompt(test_case):
    """Merges the prompt and the test case, runs it through the model, and returns the output."""
    prompt = f"""Please solve the following task: {test_case['task']}"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

## Run Test Case

In [6]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result."""
    output = run_prompt(test_case)
    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
    }

## Run Evaluation

In [7]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case for each test case, then returns the results."""
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [9]:
with open("dataset.json") as f:
    dataset = json.load(f)
results = run_eval(dataset)

In [10]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS CloudWatch Log Parser\n\nHere's a comprehensive solution with multiple approaches:\n\n## Solution 1: Simple Regex Approach\n\n```python\nimport re\nfrom typing import Dict, Tuple, Optional\n\ndef parse_cloudwatch_log(log_entry: str) -> Dict[str, str]:\n    \"\"\"\n    Parses an AWS CloudWatch log entry and extracts timestamp, level, and message.\n    \n    Format: '[TIMESTAMP] [LEVEL] Message content here'\n    \n    Args:\n        log_entry: The log entry string to parse\n        \n    Returns:\n        Dictionary with keys: 'timestamp', 'level', 'message'\n        \n    Raises:\n        ValueError: If log entry doesn't match expected format\n    \"\"\"\n    pattern = r'\\[([^\\]]+)\\]\\s+\\[([^\\]]+)\\]\\s+(.*)'\n    match = re.match(pattern, log_entry.strip())\n    \n    if not match:\n        raise ValueError(f\"Invalid log format: {log_entry}\")\n    \n    return {\n        'timestamp': match.group(1),\n        'level': match.group(2),\n        'message'